In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import os

base_path = "/mnt/data"

# Load data
employee = pd.read_csv(os.path.join(base_path, "Team_M4_employee_data_survey.csv"))
enrollment = pd.read_csv(os.path.join(base_path, "Team_M4_enrollment_data_survey.csv"))
questions = pd.read_csv(os.path.join(base_path, "Team_M4_questions.csv"))
respondent_employee = pd.read_csv(os.path.join(base_path, "Team_M4_respondent_employee.csv"))
responses = pd.read_csv(os.path.join(base_path, "Team_M4_responses.csv"))

# Merge survey responses to employees
survey_emp = responses.merge(respondent_employee, on="RespondentID", how="left")

# Merge with enrollment data to attach outcomes
full = survey_emp.merge(enrollment, on="EmployeeID", how="left")

# Create learning gain
full["LearningGain"] = full["Outcome_Proficiency"] - full["Intake_Proficiency"]

# Select motivation items
motivation_cols = ['Q1', 'Q2', 'Q3', 'Q4', 'Q5', 'Q6', 'Q7', 'Q8', 'Q9', 'Q10', 'Q11', 'Q12']

X = full[motivation_cols].copy()

# Handle any missing values (mean imputation)
X_imputed = X.fillna(X.mean())

# Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

# PCA
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

print("Explained variance ratio:", pca.explained_variance_ratio_)

# Choose first 3 PCs for clustering
n_components_for_clustering = 3
X_pca_3 = X_pca[:, :n_components_for_clustering]

# KMeans clustering with k=3
kmeans_3 = KMeans(n_clusters=3, n_init=20, random_state=424)
clusters_3 = kmeans_3.fit_predict(X_pca_3)

full["Cluster3"] = clusters_3

# Cluster summaries
cluster_summary = full.groupby("Cluster3")[motivation_cols + ["CompletionFlag", "LearningGain"]].mean()
print(cluster_summary)
